In [1]:
# Import necessary libraries for data manipulation and visualization
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load the spam.csv dataset into a pandas DataFrame
df = pd.read_csv('spam.csv', encoding='latin-1')
# Display the first few rows of the DataFrame to inspect its structure
df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [2]:
# Get a concise summary of the DataFrame, including data types and non-null values
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   v1          5572 non-null   object
 1   v2          5572 non-null   object
 2   Unnamed: 2  50 non-null     object
 3   Unnamed: 3  12 non-null     object
 4   Unnamed: 4  6 non-null      object
dtypes: object(5)
memory usage: 217.8+ KB


In [3]:
# Calculate the number of missing values for each column
df.isnull().sum()

,0
v1,0
v2,0
Unnamed: 2,5522
Unnamed: 3,5560
Unnamed: 4,5566


In [4]:
# Drop the 'Unnamed: 2', 'Unnamed: 3', and 'Unnamed: 4' columns as they contain mostly null values
df = df.drop(['Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'], axis=1)
# Display the first few rows of the DataFrame after dropping columns
df.head()

,v1,v2
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [5]:
# Rename the columns 'v1' to 'target' and 'v2' to 'text' for better readability
df.rename(columns={"v1": "target", "v2": "text"}, inplace=True)

In [6]:
# Display the first few rows of the DataFrame after renaming columns
df.head()

,target,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [7]:
# Convert the 'target' column from categorical ('ham', 'spam') to numerical (0, 1)
df["target"] = df["target"].map({"ham": 0, "spam": 1})
# Count the occurrences of each unique value in the 'target' column
df["target"].value_counts()

,count
target,
0,4825
1,747


In [8]:
# Import NLTK library for natural language processing
import nltk
# Download necessary NLTK data: stopwords (common words to be removed) and punkt (tokenizer)
nltk.download('stopwords')
nltk.download('punkt')
# Import stopwords and PorterStemmer from NLTK
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


In [10]:
# Import regular expression module
import re
# Get a set of English stopwords for efficient lookup
stop_words = set(stopwords.words('english'))
# Initialize the Porter Stemmer for word stemming
ps = PorterStemmer()

# Define a function to clean text data
def clean_text(text):
    # Convert text to lowercase
    text = text.lower()
    # Remove non-alphabetic characters and replace with spaces
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # Remove stopwords from the text
    text = ' '.join([word for word in text.split() if word not in stop_words])
    # Apply stemming to the remaining words
    text = ' '.join([ps.stem(word) for word in text.split()])
    return text

In [11]:
# Apply the clean_text function to the 'text' column to create a new 'cleanedText' column
df["cleanedText"] = df['text'].map(clean_text)
# Display the first few rows of the DataFrame, including the new 'cleanedText' column
df.head()

,target,text,cleanedText
0,0,"Go until jurong point, crazy.. Available only ...",go jurong point crazy available bugis n great ...
1,0,Ok lar... Joking wif u oni...,ok lar joking wif u oni
2,1,Free entry in 2 a wkly comp to win FA Cup fina...,free entry wkly comp win fa cup final tkts st ...
3,0,U dun say so early hor... U c already then say...,u dun say early hor u c already say
4,0,"Nah I don't think he goes to usf, he lives aro...",nah dont think goes usf lives around though


In [12]:
# Drop the original 'text' column as it has been cleaned and replaced by 'cleanedText'
df.drop(columns=['text'], inplace=True)
# Display the first few rows of the DataFrame after dropping the 'text' column
df.head()

,target,cleanedText
0,0,go jurong point crazy available bugis n great ...
1,0,ok lar joking wif u oni
2,1,free entry wkly comp win fa cup final tkts st ...
3,0,u dun say early hor u c already say
4,0,nah dont think goes usf lives around though


In [13]:
# Import train_test_split for splitting data into training and testing sets
from sklearn.model_selection import train_test_split
# Define features (X) as the 'cleanedText' and target (y) as 'target'
X = df['cleanedText']
y = df['target']
# Split the data into training and testing sets with a test size of 20%
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2)

In [14]:
# Import TfidfVectorizer for converting text data into TF-IDF features
from sklearn.feature_extraction.text import TfidfVectorizer
# Initialize the TfidfVectorizer
tfidf = TfidfVectorizer()
# Fit the TfidfVectorizer on the training data and transform it
X_train_tfidf = tfidf.fit_transform(X_train)
# Transform the test data using the fitted TfidfVectorizer
X_test_tfidf = tfidf.transform(X_test)

In [15]:
# Display the shape of the TF-IDF transformed test data
X_test_tfidf.shape

(1115, 7424)

In [16]:
# Display the shape of the TF-IDF transformed training data
X_train_tfidf.shape

(4457, 7424)

In [17]:
# Import Multinomial Naive Bayes classifier
from sklearn.naive_bayes import MultinomialNB
# Initialize and train the Multinomial Naive Bayes model
naiveB = MultinomialNB()
naiveB.fit(X_train_tfidf, y_train)
# Print the accuracy score of the model on the training data
print(naiveB.score(X_train_tfidf, y_train))
# Print the accuracy score of the model on the test data
print(naiveB.score(X_test_tfidf, y_test))

0.9777877496073593
0.9560538116591928


In [18]:
# Import classification_report for detailed evaluation metrics
from sklearn.metrics import classification_report
# Predict the target values for the test data using the trained Naive Bayes model
y_pred = naiveB.predict(X_test_tfidf)
# Print the classification report comparing actual (y_test) and predicted (y_pred) values
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.95      1.00      0.97       951
           1       1.00      0.70      0.82       164

    accuracy                           0.96      1115
   macro avg       0.98      0.85      0.90      1115
weighted avg       0.96      0.96      0.95      1115



In [19]:
# Import Logistic Regression classifier
from sklearn.linear_model import LogisticRegression

# Initialize and train the Logistic Regression model
lg = LogisticRegression()
lg.fit(X_train_tfidf, y_train)
# Print the accuracy score of the model on the training data
print(lg.score(X_train_tfidf, y_train))
# Print the classification report using the previous predictions (y_pred) for comparison
print(classification_report(y_test, y_pred))

0.9632039488445142
              precision    recall  f1-score   support

           0       0.95      1.00      0.97       951
           1       1.00      0.70      0.82       164

    accuracy                           0.96      1115
   macro avg       0.98      0.85      0.90      1115
weighted avg       0.96      0.96      0.95      1115



In [20]:
# Import Support Vector Machine (SVC) classifier
from sklearn.svm import SVC
# Initialize and train the SVM model
svm = SVC()
svm.fit(X_train_tfidf, y_train)
# Predict the target values for the test data using the trained SVM model
y_pred = svm.predict(X_test_tfidf)
# Print the classification report comparing actual (y_test) and predicted (y_pred) values
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.97      1.00      0.99       951
           1       1.00      0.84      0.91       164

    accuracy                           0.98      1115
   macro avg       0.99      0.92      0.95      1115
weighted avg       0.98      0.98      0.98      1115



In [21]:
# Import RandomForestClassifier
from sklearn.ensemble import RandomForestClassifier
# Initialize and train the Random Forest model
rf = RandomForestClassifier()
rf.fit(X_train_tfidf, y_train)
# Print the accuracy score of the model on the training data
print(rf.score(X_train_tfidf, y_train))
# Print the classification report using the previous predictions (y_pred) for comparison
print(classification_report(y_test, y_pred))

0.9997756338344178
              precision    recall  f1-score   support

           0       0.97      1.00      0.99       951
           1       1.00      0.84      0.91       164

    accuracy                           0.98      1115
   macro avg       0.99      0.92      0.95      1115
weighted avg       0.98      0.98      0.98      1115



In [22]:
# Import cross_val_score for cross-validation and numpy for numerical operations
from sklearn.model_selection import cross_val_score
import numpy as np

# Define a dictionary of models to be evaluated
models = {
    "Naive Bayes": MultinomialNB(),
    "Logistic Regression": LogisticRegression(),
    "SVM": SVC(),
    "Random Forest": RandomForestClassifier()
}

# Iterate through each model, perform 5-fold cross-validation, and print precision scores
for name, model in models.items():
    scores = cross_val_score(model, X_train_tfidf, y_train, cv=5, scoring="precision")
    print(f"{name}: {scores.mean():.3f} (+/- {scores.std():.3f})")

Naive Bayes: 1.000 (+/- 0.000)
Logistic Regression: 0.978 (+/- 0.018)
SVM: 0.996 (+/- 0.005)
Random Forest: 0.998 (+/- 0.004)


In [23]:
# Iterate through each model, perform 5-fold cross-validation, and print recall scores
for name, model in models.items():
    scores = cross_val_score(model, X_train_tfidf, y_train, cv=5, scoring="recall")
    print(f"{name}: {scores.mean():.3f} (+/- {scores.std():.3f})")

Naive Bayes: 0.701 (+/- 0.064)
Logistic Regression: 0.590 (+/- 0.046)
SVM: 0.758 (+/- 0.039)
Random Forest: 0.779 (+/- 0.042)


In [24]:
# Import XGBoost classifier
from xgboost import XGBClassifier

# Initialize the XGBoost model
xgb = XGBClassifier()
# Perform 5-fold cross-validation for precision and recall for XGBoost
scores_p = cross_val_score(xgb, X_train_tfidf, y_train, cv=5, scoring="precision")
scores_r = cross_val_score(xgb, X_train_tfidf, y_train, cv=5, scoring="recall")
# Print the mean and standard deviation of precision scores for XGBoost
print(f"XGBoost Precision: {scores_p.mean():.3f} (+/- {scores_p.std():.3f})")
# Print the mean and standard deviation of recall scores for XGBoost
print(f"XGBoost Recall: {scores_r.mean():.3f} (+/- {scores_r.std():.3f})")

XGBoost Precision: 0.929 (+/- 0.022)
XGBoost Recall: 0.823 (+/- 0.020)
